# Context Delay Metrics

Reports robust delay metrics by line, direction, local hour, and weekday/weekend context after conservative quality filtering and trip-stop bucketing.

Results come from the shared DuckDB cache under `outputs/report-cache/`, the same path the CLI scripts use by default. The first run builds it (about 1.5 minutes, DuckDB capped at 3 GB); later runs reuse it. The legacy `load_observations` path loads every observation into pandas and needs more memory than a 15 GiB machine has for the full database.

In [ ]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

spec = importlib.util.spec_from_file_location(
    "context_delay_metrics",
    PROJECT_ROOT / "analysis" / "context-delay-metrics.py",
)
context_metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context_metrics)

DB = PROJECT_ROOT / "data" / "foli.db"
LIMIT = 50
MIN_OBSERVATIONS = 50
TIMEZONE = "Europe/Helsinki"
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"
LINE_REF = None
DIRECTION_REF = None
DAY_TYPE = "all"

CACHE_DIR = PROJECT_ROOT / "outputs" / "report-cache"
MIN_TRIPS = 10

import cached_queries

In [ ]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    limit = LIMIT
    min_observations = MIN_OBSERVATIONS
    min_trips = MIN_TRIPS
    timezone = TIMEZONE
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET
    line_ref = LINE_REF
    direction_ref = DIRECTION_REF
    day_type = DAY_TYPE

cached_queries.context_delay_metrics(Args)